# Coorte e deriva

Aqui estão as internações e os óbitos de cada ano de 2019 a 2025, a deriva trimestral do X1 e a conferência dos dados de
2026 que fiz antes da rodada confirmatória. As contagens saem direto das bases de `data/processed/`; da base lacrada,
`sih_norte_minas_selado.parquet`, o notebook lê só as colunas DT_INTER, ANO_CMPT e MES_CMPT. As séries do X1 estão em
`reports/exploratorios/morte/nenhuma/` (`x1_deriva_trimestral.csv` e `resultados_x1.json`), e o que a rodada registrou,
em `reports/confirmatorio/x7a.json`.

In [1]:
import json

import pandas as pd

from triagem_baixo_risco import config
from triagem_baixo_risco.dataset import admission_dates, sealed_path
from triagem_baixo_risco.features import build_strata

EXPLORATORIOS = config.REPORTS_DIR / "exploratorios" / "morte" / "nenhuma"


def pct(x, casas=2):
    return f"{100 * x:.{casas}f}%".replace(".", ",")


def num(x, casas=0):
    # 1234567.891 com casas=2 vira "1.234.567,89"
    return f"{x:,.{casas}f}".replace(",", " ").replace(".", ",").replace(" ", ".")

## Internações e óbitos por ano

A coorte vai pela data de internação, a partir de 01/01/2019. A base aberta também tem 6.573 AIH de internações mais
antigas, cobradas em competências de 2019 em diante, e essas ficam de fora.

In [2]:
aberta = pd.read_parquet(config.OPEN_DATA, columns=["DT_INTER", "MORTE", "ANO_CMPT", "MES_CMPT"])
aberta["data"] = admission_dates(aberta)
antes = aberta["data"] < pd.Timestamp(config.COHORT_START)
print("linhas da base aberta:", num(len(aberta)))
print("internações anteriores a 2019:", num(antes.sum()))
print("linhas sem data de internação válida:", num(aberta["data"].isna().sum()))
coorte = aberta[~antes]

linhas da base aberta: 825.434
internações anteriores a 2019: 6.573
linhas sem data de internação válida: 0


In [3]:
ano = coorte["data"].dt.year
obito = coorte["MORTE"].str.strip() == "1"
por_ano = pd.DataFrame({"internações": ano.value_counts(), "óbitos": obito.groupby(ano).sum()}).sort_index()
por_ano.loc["2019-2025"] = por_ano.sum()
pd.DataFrame(
    {
        "internações": por_ano["internações"].map(num),
        "óbitos": por_ano["óbitos"].map(num),
        "mortalidade": (por_ano["óbitos"] / por_ano["internações"]).map(pct),
    }
).rename_axis("ano")

,internações,óbitos,mortalidade
ano,,,
2019,105.884,4.658,"4,40%"
2020,88.065,4.780,"5,43%"
2021,96.760,6.208,"6,42%"
2022,111.873,5.323,"4,76%"
2023,127.377,5.483,"4,30%"
2024,142.871,5.648,"3,95%"
2025,146.031,5.684,"3,89%"
2019-2025,818.861,37.784,"4,61%"


A mortalidade hospitalar subiu de 4,40% em 2019 para 6,42% em 2021 e caiu para 3,89% em 2025, abaixo da de antes da
pandemia. O número de internações foi no sentido contrário: depois do mínimo de 88.065 em 2020, chegou a 146.031 em
2025. No X2, por exemplo, o modelo aprende com 2019 a 2021, anos de mortalidade entre 4,40% e 6,42%, e é aplicado a
2023-2025, com 4,30% ou menos.

## Hospitais, municípios e residência

O filtro do ETL usa o município do hospital. Aqui conto os hospitais (CNES) e os municípios de atendimento com
alguma internação de 2019 a 2025, e quantas internações eram de moradores de fora dos 89 municípios da região.

In [4]:
LOCAL = [
    "CNES",
    "MUNIC_MOV",
    "MUNIC_RES",
    "DIAS_PERM",
    "IDADE",
    "COD_IDADE",
    "COBRANCA",
    "DIAG_PRINC",
    "CAR_INT",
    "SEXO",
]
mais = pd.read_parquet(config.OPEN_DATA, columns=LOCAL).loc[coorte.index]
municipios = pd.read_csv(config.MUNICIPIOS_CSV, dtype=str)
de_fora = ~mais["MUNIC_RES"].str[:6].isin(municipios["cod6"])
pd.Series(
    {
        "hospitais (CNES)": num(mais["CNES"].nunique()),
        "municípios de atendimento": num(mais["MUNIC_MOV"].nunique()),
        "internações de moradores de fora": pct(de_fora.mean()),
    }
).to_frame("2019-2025")

,2019-2025
hospitais (CNES),42
municípios de atendimento,30
internações de moradores de fora,"1,36%"


Foram 42 hospitais em 30 municípios, e 1,36% das internações eram de moradores de fora da região.

## Obstétricas e pediátricas

A coorte inclui as internações obstétricas, marcadas como em `docs/dados.md`, e as de pacientes com menos de 18 anos.
Na tabela, as pediátricas são as de menores de 18 anos que não são obstétricas.

In [5]:
estratos = build_strata(mais)
grupo = pd.Series("demais", index=coorte.index)
grupo[(estratos["idade"] < 18) & ~estratos["obstetrica"]] = "pediátricas"
grupo[estratos["obstetrica"]] = "obstétricas"
grupos = pd.DataFrame({"internações": grupo.value_counts(), "óbitos": obito.groupby(grupo).sum()})
pd.DataFrame(
    {
        "internações": grupos["internações"].map(num),
        "parte da coorte": (grupos["internações"] / len(coorte)).map(lambda v: pct(v, 1)),
        "óbitos": grupos["óbitos"].map(num),
        "mortalidade": (grupos["óbitos"] / grupos["internações"]).map(pct),
    }
).loc[["obstétricas", "pediátricas", "demais"]]

,internações,parte da coorte,óbitos,mortalidade
obstétricas,145.539,"17,8%",68,"0,05%"
pediátricas,113.446,"13,9%",1.274,"1,12%"
demais,559.876,"68,4%",36.442,"6,51%"


As obstétricas são 17,8% da coorte e tiveram 68 óbitos, 0,05%. Nas pediátricas, a mortalidade foi de 1,12%, e nas
demais, de 6,51%.

## O atraso de cobrança no fim de 2025

Uma AIH só é cobrada depois que a internação termina, então as longas chegam mais tarde à base. Filtrando as
competências até dez/2025, a base volta ao que eu tinha antes de baixar as competências de 2026, e dá para comparar
as internações de dezembro de 2025 nos dois momentos.

In [6]:
cobrada_ate_dez = (coorte["ANO_CMPT"] + coorte["MES_CMPT"]).astype(int) <= 202512
longa = pd.to_numeric(mais["DIAS_PERM"], errors="coerce") >= 8
de_dezembro = coorte["data"] >= pd.Timestamp("2025-12-01")
jan_a_nov = coorte["data"].dt.year.eq(2025) & ~de_dezembro
recortes = {
    "dez/2025, competências até dez/2025": de_dezembro & cobrada_ate_dez,
    "jan a nov/2025, competências até dez/2025": jan_a_nov & cobrada_ate_dez,
    "dez/2025, competências até jul/2026": de_dezembro,
}
pd.DataFrame(
    {
        nome: {
            "internações": num(m.sum()),
            "óbitos": pct(obito[m].mean(), 1),
            "8 dias ou mais": pct(longa[m].mean(), 1),
        }
        for nome, m in recortes.items()
    }
).T

,internações,óbitos,8 dias ou mais
"dez/2025, competências até dez/2025",7.402,"2,7%","6,9%"
"jan a nov/2025, competências até dez/2025",132.245,"3,9%","13,4%"
"dez/2025, competências até jul/2026",12.277,"3,8%","13,9%"


Com as competências até dez/2025, dezembro tinha 7.402 internações, com 2,7% de óbitos e 6,9% de permanências de 8
dias ou mais, contra 3,9% e 13,4% de janeiro a novembro. Com as competências até jul/2026, dezembro passou a 12.277
internações e 3,8% de óbitos. As internações longas chegam mais tarde, e por isso a coorte vai pela data de
internação, com o desfecho buscado em todas as competências publicadas.

## Deriva por trimestre

Cada linha do X1 é um trimestre de internação, e as colunas de óbito a raça sem informação são porcentagens das
internações do trimestre. Respiratória é o capítulo X da CID-10; COVID, os CID B34 e U07; raça sem informação, RACA_COR
igual a 99. Hospitais conta os CNES com alguma internação no trimestre. Urgência, obstétrica e censurada seguem os
marcadores de `docs/dados.md`.

In [7]:
x1 = pd.read_csv(EXPLORATORIOS / "x1_deriva_trimestral.csv", index_col="trimestre")
fracoes = {
    "óbito": "obito",
    "urgência": "urgencia",
    "obstétrica": "obstetrica",
    "censurada": "censurada",
    "respiratória": "respiratorio",
    "COVID": "covid",
    "raça sem informação": "raca_sem_info",
}
serie = pd.DataFrame({nome: x1[coluna].map(lambda v: pct(v, 1)) for nome, coluna in fracoes.items()})
serie.insert(0, "internações", x1["internacoes"].map(num))
serie.insert(3, "idade média", x1["idade"].map(lambda v: num(v, 1)))
serie["hospitais"] = x1["hospitais_ativos"]
serie

,internações,óbito,urgência,idade média,obstétrica,censurada,respiratória,COVID,raça sem informação,hospitais
trimestre,,,,,,,,,,
2019T1,25.281,"4,3%","85,5%","42,7","22,8%","10,1%","7,8%","0,0%","28,5%",35
2019T2,27.357,"4,3%","85,1%","41,4","20,8%","8,8%","11,4%","0,0%","27,7%",36
2019T3,27.509,"4,4%","83,3%","43,4","20,3%","9,2%","10,7%","0,0%","29,4%",34
2019T4,25.737,"4,6%","85,6%","43,1","19,9%","9,6%","9,2%","0,0%","28,8%",35
2020T1,24.289,"4,3%","88,2%","43,2","22,1%","10,5%","9,2%","0,0%","28,4%",35
2020T2,19.828,"5,5%","96,6%","43,0","27,6%","12,1%","6,2%","1,3%","29,3%",36
2020T3,21.203,"6,3%","96,1%","44,1","24,6%","11,3%","6,4%","4,1%","28,1%",36
2020T4,22.745,"5,7%","92,1%","43,8","22,6%","11,6%","5,5%","4,1%","29,0%",36
2021T1,22.740,"8,0%","95,4%","44,1","24,3%","12,0%","5,4%","11,7%","32,9%",37


O pior trimestre foi o 1º de 2021, com 8,0% de óbitos, quando 11,7% das internações tinham CID de COVID; no seguinte,
essa parte chegou a 15,5%. Fora da pandemia a mudança foi mais lenta, e não voltou atrás. As urgências passaram de 85,5%
das internações no 1º trimestre de 2019 para 74,1% no último de 2025. No mesmo intervalo, a parte obstétrica caiu de
22,8% para 12,1%, e a idade média subiu pouco mais de três anos, de 42,7 para 45,9. A raça sem informação oscilou entre
27,7% e 35,2% por trimestre até 2021 e zerou a partir do 2º trimestre de 2023, uma mudança de codificação que tirou a
variável do modelo.

O X1 também treina, com as variáveis do modelo, um classificador que separa as internações de cada ano das de 2019. Uma
AUROC de 0,5 diria que os dois anos não se distinguem.

In [8]:
dominio = json.loads((EXPLORATORIOS / "resultados_x1.json").read_text())["auroc_dominio"]
pd.DataFrame({"AUROC contra 2019": {ano: num(v, 3) for ano, v in dominio.items()}}).rename_axis("ano")

,AUROC contra 2019
ano,
2020,"0,659"
2021,"0,706"
2022,"0,669"
2023,"0,697"
2024,"0,726"
2025,"0,741"


A AUROC foi de 0,659 em 2020 a 0,741 em 2025 e cresce todo ano desde 2022. O perfil de quem se interna continuou se
afastando do de 2019 depois da pandemia.

## Os dados de 2026

Antes da rodada confirmatória, conferi se as internações de 2026 estavam chegando completas, comparando o volume de cada
mês com o do mesmo mês de 2025. Para os meses mais recentes, a comparação é desigual: as AIH de 2025 já tiveram todas as
competências até jul/2026 para ser cobradas, e as de julho de 2026, só a do próprio mês.

In [9]:
lacrada = pd.read_parquet(sealed_path(config.OPEN_DATA), columns=["DT_INTER", "ANO_CMPT", "MES_CMPT"])
lacrada["data"] = admission_dates(lacrada)
de_2025 = coorte["data"].dt.year == 2025
volume = pd.DataFrame(
    {
        "2025": coorte.loc[de_2025, "data"].dt.month.value_counts(),
        "2026": lacrada["data"].dt.month.value_counts(),
    }
).dropna()
LIMITE = 0.90
razao = volume["2026"] / volume["2025"]
pd.DataFrame(
    {
        "internações em 2025": volume["2025"].map(num),
        "internações em 2026": volume["2026"].map(num),
        "razão": razao.map(lambda v: num(v, 2)),
        "abaixo de 0,90": (razao < LIMITE).map({True: "sim", False: "não"}),
    }
).rename_axis("mês")

,internações em 2025,internações em 2026,razão,"abaixo de 0,90"
mês,,,,
1,11.509,12.199,"1,06",não
2,10.954,11.080,"1,01",não
3,11.731,13.342,"1,14",não
4,12.352,12.902,"1,04",não
5,12.596,12.978,"1,03",não
6,12.185,10.431,"0,86",sim
7,13.024,6.305,"0,48",sim


Janeiro a março, os meses do teste de X7a e X7b, ficaram entre 1,01 e 1,14 do volume de 2025, acima do limite de 0,90
que eu tinha fixado para apontar seguimento incompleto, e abril e maio, em 1,04 e 1,03. Junho, com 0,86, só tinha passado por duas competências, e ele entra no teste do X7c, que espera as competências
até out/2026. Julho, com 0,48, não entra em rodada nenhuma.

In [10]:
def competencia(base):
    return base["ANO_CMPT"] + "-" + base["MES_CMPT"]


presentes = set(competencia(coorte)) | set(competencia(lacrada))
esperadas = [str(m) for m in pd.period_range("2019-01", "2026-07", freq="M")]
ausentes = [m for m in esperadas if m not in presentes]
print(f"competências de 2019-01 a 2026-07: {len(esperadas)}; ausentes: {', '.join(ausentes) or 'nenhuma'}")
mes = lacrada["data"].dt.strftime("%Y-%m")
triangulo = pd.crosstab(mes, competencia(lacrada), margins=True, margins_name="total")
triangulo.map(num).rename_axis(index="mês de internação", columns="competência")

competências de 2019-01 a 2026-07: 91; ausentes: nenhuma


competência,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,total
mês de internação,,,,,,,,
2026-01,7.040,3.579,1.333,234,8,5,0,12.199
2026-02,0,6.333,3.346,1.166,230,5,0,11.080
2026-03,0,0,7.319,4.444,1.234,324,21,13.342
2026-04,0,0,0,7.546,3.740,1.317,299,12.902
2026-05,0,0,0,0,7.254,4.003,1.721,12.978
2026-06,0,0,0,0,0,6.437,3.994,10.431
2026-07,0,0,0,0,0,0,6.305,6.305
total,7.040,9.912,11.998,13.390,12.466,12.091,12.340,79.237


Nenhuma das 91 competências falta. Na tabela, cada linha é um mês de internação e cada coluna, a competência em que a
AIH foi cobrada. Das 12.199 internações de janeiro, 7.040 entraram na própria competência de janeiro, e só 13 depois de
abril, 8 em maio e 5 em junho.

## O que a rodada confirmatória registrou

Os óbitos de 2026 entram neste notebook só agora, pelo registro do X7a, que também guarda o que a rodada conferiu antes
de ajustar qualquer modelo.

In [11]:
x7a = json.loads((config.REPORTS_DIR / "confirmatorio" / "x7a.json").read_text())
integridade, competencias = x7a["integridade"], x7a["competencias"]
pd.Series(
    {
        "linhas lacradas": num(integridade["linhas_lacradas"]),
        "N_AIH únicos nas duas bases": num(integridade["n_aih_unicos"]),
        "IDENT": integridade["ident"],
        "competências usadas": f"{competencias['primeira']} a {competencias['ultima']}",
        "linhas fora do corte de competência": num(competencias["linhas_fora_do_corte"]),
        "internações no teste (jan-mar/2026)": num(x7a["n_teste"]),
        "óbitos no teste": num(x7a["obitos_teste"]),
        "mortalidade no teste": pct(x7a["obitos_teste"] / x7a["n_teste"]),
    }
).to_frame("X7a")

,X7a
linhas lacradas,79.237
N_AIH únicos nas duas bases,898.098
IDENT,todas 1
competências usadas,2019-01 a 2026-07
linhas fora do corte de competência,0
internações no teste (jan-mar/2026),36.621
óbitos no teste,1.345
mortalidade no teste,"3,67%"


Juntando as duas bases, a rodada achou 898.098 N_AIH, as 818.861 internações de 2019 a 2025 mais as 79.237 lacradas,
nenhum repetido e todos com IDENT igual a 1. As 36.621 internações do teste são a soma de janeiro a março na tabela de
volume. Delas, 1.345 terminaram em óbito, 3,67%, abaixo dos 3,89% de 2025.